In [ ]:
# ==============================================================================
# 1_data_ingestion.ipynb
# ==============================================================================
import os
import pandas as pd
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType

# Mount Google Drive
drive.mount('/content/drive')

# Setup SparkSession
spark = SparkSession.builder \
    .appName("HIGGS_Classification_Master") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("\n>>> STEP 2: RDD-Based Parallel Ingestion (Bronze Layer)...")
path_higgs = "/content/drive/MyDrive/Bigdata/HIGGS/HIGGS.csv"

# 1. RDD Ingestion
sc = spark.sparkContext
raw_rdd = sc.textFile(path_higgs, minPartitions=200)

# 2. Parse CSV using RDD map function
def parse_row(row):
    fields = row.split(",")
    return tuple([float(x) for x in fields])

parsed_rdd = raw_rdd.map(parse_row)

# 3. Convert RDD to DataFrame (Silver Layer)
schema = StructType([StructField("label", DoubleType(), True)] +
                    [StructField(f"feature_{i}", DoubleType(), True) for i in range(1, 29)])

df_raw = spark.createDataFrame(parsed_rdd, schema)
print(f"   Total Rows Loaded: {df_raw.count()} | Total Columns: {len(df_raw.columns)}")